# NB-302: Transformer Architecture

Build a complete Transformer from scratch.

## Learning Objectives
- Positional encoding
- Encoder-decoder architecture
- Complete transformer block
- Language modeling

In [ ]:
import torch
import torch.nn as nn
import math

print("Ready to build Transformers")

## 1. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # TODO: Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Test
pe = PositionalEncoding(d_model=512)
print(f"Positional encoding shape: {pe.pe.shape}")

## 2. Transformer Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        # TODO: Self-attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        # Feed-forward
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        # Layer norms and dropout
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # TODO: Self-attention block with residual
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        # Feed-forward block with residual
        ff_output = self.feed_forward(x)
        x = x + self.dropout2(ff_output)
        x = self.norm2(x)

        return x

# Test
encoder_block = TransformerEncoderBlock(d_model=512, n_heads=8, d_ff=2048)
x = torch.randn(2, 10, 512)
output = encoder_block(x)
print(f"Encoder output shape: {output.shape}")

## 3. Complete Transformer

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6, d_ff=2048, max_len=512):
        super().__init__()

        # TODO: Embedding and positional encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        # TODO: Encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])

        # Output layer
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        # Embed
        x = self.embedding(x)
        x = self.pos_encoding(x)

        # TODO: Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x, mask)

        # Output projection
        logits = self.output_layer(x)

        return logits

# Test
model = Transformer(vocab_size=10000, d_model=256, n_heads=4, n_layers=2)
input_ids = torch.randint(0, 10000, (2, 20))
output = model(input_ids)
print(f"Output shape: {output.shape}")

## Exercise

1. Add causal masking for GPT-style decoder
2. Implement cross-attention for encoder-decoder
3. Add gradient clipping